<a href="https://colab.research.google.com/github/Mayuri0320/Datax504-Mayuri/blob/main/a3_regularisation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 3 — IMDB Sentiment & Regularisation

**DATAX504** · Due end of Week 6

Build on **Chapter 5** (regularisation / early stopping) and **Chapter 7** (Keras `compile` / `fit`, callbacks, saving). A compact Sequential stack is fine; you may use the **Functional API** if you prefer (same depth of learning either way).

## Tasks
1. Load IMDB, pad sequences, create a train / validation split
2. Train a **baseline** sentiment model and record best validation accuracy
3. Produce a **diagram** of your model architecture (see §2b)
4. Apply **two** of: L2, dropout, early stopping — report best val accuracy and name the techniques
5. Learning-rate schedule experiment (`ReduceLROnPlateau` or `LearningRateScheduler`)
6. Save the model you consider best with `model.save(...)` and check that it reloads
7. ~200-word reflection on what you learned

## Moodle fields
`a3_repo`, `a3_baseline_val`, `a3_best_val`, `a3_techniques`, `a3_model_file`

## AI disclosure (required)
Fill the table in the next cell before you submit.

## AI disclosure

| Field | Your answer |
|-------|-------------|
| Tool / model | Chatgpt|
| Used for | understanding of L2 regularisation and dropout and how they work|
| Verified how | Ran every cell and ensured all cells matched the assignment requirements|
| Not used for | Testing|

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import keras
from keras import layers, regularizers, callbacks

# Optional for plot_model (Ch 7). If import fails, use model.summary() + a drawn sketch (see §2b).
try:
    from keras.utils import plot_model
except ImportError:
    plot_model = None

In [2]:
max_features = 10000
max_len = 500

(x_train, y_train), (x_test, y_test) = keras.datasets.imdb.load_data(num_words=max_features)

# TODO: pad sequences so every review has length max_len
x_train = keras.utils.pad_sequences(x_train, maxlen=max_len)
x_test = keras.utils.pad_sequences(x_test, maxlen=max_len)

# TODO: validation split — first 10_000 samples for val, remainder for train
x_val = x_train[:10000]
y_val = y_train[:10000]

x_train = x_train[10000:]
y_train = y_train[10000:]

print("train:", getattr(x_train, "shape", None), getattr(y_train, "shape", None))
print("val:  ", getattr(x_val, "shape", None), getattr(y_val, "shape", None))
print("test: ", getattr(x_test, "shape", None), getattr(y_test, "shape", None))

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
train: (15000, 500) (15000,)
val:   (10000, 500) (10000,)
test:  (25000, 500) (25000,)


## 2. Baseline model

Suggested architecture (you may change widths if you document why):
1. `Embedding(max_features, 32, input_length=max_len)` — turns token ids into vectors
2. `GlobalAveragePooling1D()` — mean over the time axis
3. `Dense(32, activation="relu")`
4. `Dense(1, activation="sigmoid")` — binary sentiment

Then:
- `compile` with a suitable optimizer, **binary cross-entropy** loss, and accuracy metric
- `fit` with `validation_data=(x_val, y_val)` for ~10 epochs (batch size is your choice; 512 is common for this data)
- record the **best** validation accuracy over epochs for Moodle `a3_baseline_val`

In [3]:
def build_baseline():
    # Build Sequential baseline model
    model = keras.Sequential([
        layers.Embedding(max_features, 32, input_length=max_len),
        layers.GlobalAveragePooling1D(),
        layers.Dense(32, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ])

    # Compile model
    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model


baseline = build_baseline()

# Fit baseline model
hist_baseline = baseline.fit(
    x_train, y_train,
    epochs=10,
    batch_size=512,
    validation_data=(x_val, y_val),
    verbose=1,
)

# Best validation accuracy across epochs
baseline_val_acc = max(hist_baseline.history["val_accuracy"])
print(f"Baseline best val accuracy (Moodle a3_baseline_val): {baseline_val_acc:.4f}")

Epoch 1/10


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


30/30 ━━━━━━━━━━━━━━━━━━━━ 6s 145ms/step - accuracy: 0.5292 - loss: 0.6920 - val_accuracy: 0.5856 - val_loss: 0.6897
Epoch 2/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 4s 99ms/step - accuracy: 0.6107 - loss: 0.6859 - val_accuracy: 0.6753 - val_loss: 0.6801
Epoch 3/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 3s 101ms/step - accuracy: 0.6745 - loss: 0.6685 - val_accuracy: 0.6306 - val_loss: 0.6565
Epoch 4/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 3s 98ms/step - accuracy: 0.6963 - loss: 0.6342 - val_accuracy: 0.6309 - val_loss: 0.6243
Epoch 5/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 5s 167ms/step - accuracy: 0.7098 - loss: 0.5901 - val_accuracy: 0.7672 - val_loss: 0.5676
Epoch 6/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 3s 105ms/step - accuracy: 0.7875 - loss: 0.5335 - val_accuracy: 0.7935 - val_loss: 0.5151
Epoch 7/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 3s 102ms/step - accuracy: 0.8026 - loss: 0.4858 - val_accuracy: 0.7849 - val_loss: 0.4812
Epoch 8/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 4s 127ms/step - accuracy: 0.8201 - loss: 0.4459 - val_accuracy: 0.7800 - val_lo

## 2b. Diagram of your architecture (required)

You *can* do this with the tools from **Chapter 7 / Week 5**:

```python
keras.utils.plot_model(
    baseline,                      # or your regularised model
    to_file="imdb_model.png",      # commit this image to your repo
    show_shapes=True,
    show_layer_names=True,
)
```

Also print `model.summary()` so layer shapes appear in the notebook.

**If `plot_model` fails** (missing Graphviz / pydot on your machine):
1. Keep a full `model.summary()` output in the notebook, and
2. Add a short markdown diagram or hand-drawn photo of the stack (Embedding → … → sigmoid).

Either way, a reader should see the full forward path without reading all of your code.

In [4]:
baseline.summary()

DIAGRAM_FILE = "imdb_model.png"

if plot_model is not None:
    try:
        plot_model(
            baseline,
            to_file=DIAGRAM_FILE,
            show_shapes=True,
            show_layer_names=True,
        )
        print(f"Wrote {DIAGRAM_FILE} — commit it to your GitHub repo")



    except Exception as e:
        print("plot_model failed — use summary + hand diagram:", e)
else:
    print("plot_model unavailable — use model.summary() and a markdown/hand diagram")

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 500, 32)        │       320,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 32)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 963,269 (3.67 MB)

 Trainable params: 321,089 (1.22 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 642,180 (2.45 MB)

Wrote imdb_model.png — commit it to your GitHub repo


## 3. Regularised model

Choose **exactly two** (or more if curious, but Moodle asks for two) of:

| Technique | Typical hook |
|-----------|----------------|
| L2 | `kernel_regularizer=regularizers.l2(...)` on a Dense layer |
| Dropout | `layers.Dropout(rate)` after a Dense / before the output |
| Early stopping | `callbacks.EarlyStopping(monitor="val_loss", patience=..., restore_best_weights=True)` |

Hints:
- Set boolean flags below so you can print which techniques you used
- `compile` again after building the stack
- Pass a **list of callbacks** into `fit` (empty list if you did not pick early stopping)
- Train long enough that early stopping might fire (e.g. more epochs than the baseline)
- Best val accuracy → Moodle `a3_best_val`; names → `a3_techniques`

In [5]:
USE_L2 = True
USE_DROPOUT = True
USE_EARLY_STOP = False

l2_reg = regularizers.l2(1e-4)

reg_model = keras.Sequential([
    layers.Embedding(max_features, 32, input_length=max_len),
    layers.GlobalAveragePooling1D(),
    layers.Dense(
        32,
        activation="relu",
        kernel_regularizer=l2_reg if USE_L2 else None
    ),
])

if USE_DROPOUT:
    reg_model.add(layers.Dropout(0.5))

reg_model.add(layers.Dense(1, activation="sigmoid"))

# Compile regularised model
reg_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# Callbacks
cb = []

if USE_EARLY_STOP:
    cb.append(
        callbacks.EarlyStopping(
            monitor="val_loss",
            patience=2,
            restore_best_weights=True
        )
    )

# Train for longer than the baseline
hist_reg = reg_model.fit(
    x_train,
    y_train,
    epochs=15,
    batch_size=512,
    validation_data=(x_val, y_val),
    callbacks=cb,
    verbose=1,
)

# Best validation accuracy across epochs
best_val_acc = max(hist_reg.history["val_accuracy"])

techniques = []
if USE_L2:
    techniques.append("L2")
if USE_DROPOUT:
    techniques.append("dropout")
if USE_EARLY_STOP:
    techniques.append("early stopping")

print(f"Techniques (Moodle a3_techniques): {', '.join(techniques)}")
print(f"Best val accuracy (Moodle a3_best_val): {best_val_acc:.4f}")

assert len(techniques) >= 2, "Enable (at least) two techniques for full credit"

Epoch 1/15


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


30/30 ━━━━━━━━━━━━━━━━━━━━ 5s 108ms/step - accuracy: 0.5181 - loss: 0.6953 - val_accuracy: 0.5284 - val_loss: 0.6935
Epoch 2/15
30/30 ━━━━━━━━━━━━━━━━━━━━ 3s 102ms/step - accuracy: 0.5751 - loss: 0.6895 - val_accuracy: 0.7093 - val_loss: 0.6846
Epoch 3/15
30/30 ━━━━━━━━━━━━━━━━━━━━ 6s 133ms/step - accuracy: 0.6213 - loss: 0.6772 - val_accuracy: 0.7215 - val_loss: 0.6659
Epoch 4/15
30/30 ━━━━━━━━━━━━━━━━━━━━ 3s 106ms/step - accuracy: 0.6827 - loss: 0.6506 - val_accuracy: 0.6610 - val_loss: 0.6352
Epoch 5/15
30/30 ━━━━━━━━━━━━━━━━━━━━ 3s 102ms/step - accuracy: 0.7034 - loss: 0.6139 - val_accuracy: 0.7180 - val_loss: 0.5922
Epoch 6/15
30/30 ━━━━━━━━━━━━━━━━━━━━ 6s 122ms/step - accuracy: 0.7551 - loss: 0.5670 - val_accuracy: 0.8049 - val_loss: 0.5356
Epoch 7/15
30/30 ━━━━━━━━━━━━━━━━━━━━ 4s 145ms/step - accuracy: 0.7847 - loss: 0.5176 - val_accuracy: 0.7552 - val_loss: 0.5098
Epoch 8/15
30/30 ━━━━━━━━━━━━━━━━━━━━ 3s 100ms/step - accuracy: 0.7977 - loss: 0.4798 - val_accuracy: 0.7657 - val_

## 4. Learning-rate schedule

Week 5 / Ch 7 callback: try **`ReduceLROnPlateau`** (shrink LR when `val_loss` stalls)
or a custom **`LearningRateScheduler`**.

Hints:
- Start from the baseline or your regularised architecture (`build_baseline()` is fine)
- Put the schedule callback in the `callbacks=[...]` list for `fit`
- After training, plot the learning-rate history if present (`hist.history.get("lr", [])` for ReduceLROnPlateau)
- Short markdown below: which schedule, why those hyperparameters, what you observed

In [6]:
lr_model = build_baseline()

# Reduce learning rate when validation loss stops improving
lr_cb = callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=1,
    min_lr=1e-6,
    verbose=1
)

hist_lr = lr_model.fit(
    x_train,
    y_train,
    epochs=10,
    batch_size=512,
    validation_data=(x_val, y_val),
    callbacks=[lr_cb],
    verbose=1,
)

# Plot learning-rate history when available
lrs = hist_lr.history.get("lr", [])

if lrs:
    plt.figure(figsize=(8, 5))
    plt.plot(lrs, marker="o")
    plt.xlabel("Epoch")
    plt.ylabel("Learning rate")
    plt.title("Learning Rate Schedule")
    plt.grid(True)
    plt.show()
else:
    print("Learning-rate history is not available in this Keras version.")

Epoch 1/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 6s 136ms/step - accuracy: 0.5179 - loss: 0.6924 - val_accuracy: 0.5321 - val_loss: 0.6912 - learning_rate: 0.0010
Epoch 2/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 4s 103ms/step - accuracy: 0.5506 - loss: 0.6876 - val_accuracy: 0.5926 - val_loss: 0.6831 - learning_rate: 0.0010
Epoch 3/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 6s 122ms/step - accuracy: 0.6332 - loss: 0.6735 - val_accuracy: 0.6974 - val_loss: 0.6606 - learning_rate: 0.0010
Epoch 4/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 3s 111ms/step - accuracy: 0.6815 - loss: 0.6428 - val_accuracy: 0.7262 - val_loss: 0.6233 - learning_rate: 0.0010
Epoch 5/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 5s 102ms/step - accuracy: 0.7264 - loss: 0.5978 - val_accuracy: 0.7028 - val_loss: 0.5832 - learning_rate: 0.0010
Epoch 6/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 6s 148ms/step - accuracy: 0.7611 - loss: 0.5478 - val_accuracy: 0.7863 - val_loss: 0.5257 - learning_rate: 0.0010
Epoch 7/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 3s 98ms/step - accuracy: 0.7877 - loss: 0.4995 - v

**LR experiment notes** (a few sentences):

- Schedule used:
- Observed effect on val curves / final accuracy:
- Would you keep this schedule for a real project?

## 5. Save best model

Hints (Ch 7):
- Pick one of baseline / regularised / LR-tuned
- `best_model.save("something.keras")` — full architecture + weights
- `keras.models.load_model(...)` and `evaluate` on **test** once (do not retune on test)
- Commit the `.keras` file and the diagram PNG to your GitHub repo
- Filename → Moodle `a3_model_file`

In [8]:
MODEL_FILE = "imdb_sentiment_best.keras"  # Moodle a3_model_file

# Compare validation performance and select the best model
lr_best_val_acc = max(hist_lr.history["val_accuracy"])

model_candidates = {
    "baseline": (baseline_val_acc, baseline),
    "regularised": (best_val_acc, reg_model),
    "lr_tuned": (lr_best_val_acc, lr_model),
}

best_model_name, (selected_val_acc, best_model) = max(
    model_candidates.items(),
    key=lambda item: item[1][0]
)

print(
    f"Selected model: {best_model_name} "
    f"(best val accuracy: {selected_val_acc:.4f})"
)

# Save the full model
best_model.save(MODEL_FILE)

# Reload and evaluate on the test set once
loaded = keras.models.load_model(MODEL_FILE)
_, test_acc = loaded.evaluate(x_test, y_test, verbose=0)

print(f"Saved model filename (Moodle a3_model_file): {MODEL_FILE}")
print(f"Test accuracy of saved model: {test_acc:.4f}")

Selected model: regularised (best val accuracy: 0.8726)
Saved model filename (Moodle a3_model_file): imdb_sentiment_best.keras
Test accuracy of saved model: 0.8699


## 6. What did you learn?

This assignment explores the effects of regularization and of different ways of training a sentiment classification model. It starts with a simple IMDB model, which is made up of an embedding layer, a global average pooling layer and a few dense layers. This model is used as a baseline to compare the results of the other experiments. I used L2 regularisation and dropout to try to reduce overfitting of the model. The regularised model achieved a best validation accuracy of 0.8726, this shows that regularisation can improve or even slightly reduce validation performance. Things like regularisation, checking validation performance, and controlling the learning rate can have a big effect on how well the model works on new data.

*Your reflection (~200 words).*